In [1]:
import requests

In [2]:
def consume_llm_api(prompt,messages=None):
    """
    Sends a prompt to the LLM API and processes the streamed response.
    """
    url = "http://127.0.0.1:6000/api/llm-response"
    headers = {"Content-Type": "application/json"}
    payload = {"prompt": prompt,"extension":"hi"}


    response = requests.post(url, json=payload, headers=headers)
    result = response.json()
    return result.get("text", "")
        

In [8]:
prompt ="Design a visually engaging document about protecting the environment. Use a modern, eco-friendly theme with green and blue color accents, clean typography, and icons of trees, water, and recycling. Include sections for Introduction, Key Environmental Issues (climate change, pollution, deforestation), Solutions (renewable energy, conservation, sustainable living), and a Call to Action. Make it look professional yet inspiring, with infographic-style visuals and a motivational tone in Markdown (.md) format, with no additional text outside the Markdown code."
value  = consume_llm_api(prompt,messages=None)

In [10]:
print(value.split("```markdown")[-1].split("```")[0])


# 🌿 EARTH VITALITY: A Blueprint for Protection

> **Theme:** Modern Eco-Friendly | **Palette:** Forest Green & Ocean Blue  
> **Goal:** Inspire Action, Protect Tomorrow  

---

## 1. Introduction: The Pulse of Planet Earth

The health of our planet is inextricably linked to the health of humanity. We stand at a critical juncture where every decision made today ripples through the future. 

**Our Mission:** To transition from passive observation to active stewardship. By integrating nature into our modern systems, we can build a resilient world that thrives on sustainability rather than depletion.

---

## 2. Key Environmental Issues ⚠️

Understanding the threat is the first step toward healing. Below are the three pillars of environmental instability we must address immediately.

| Issue Icon 🌳 | **Climate Change** 🔥 |
| :--- | :--- |
| **The Threat** | Rising global temperatures, extreme weather events, and shifting ecosystems. |
| **Impact** | Melting ice caps, erratic agriculture, 

In [22]:
"""
md_to_pdf — Convert Markdown text directly to PDF with GitHub-style styling

This module provides an easy-to-use function to convert Markdown text strings 
directly into beautifully styled PDF documents, mimicking the VS Code "Markdown 
PDF" extension look and feel.

Usage:
    from md_to_pdf import convert_md_to_pdf

    # Convert markdown text directly (no file needed)
    pdf_path = convert_md_to_pdf(
        markdown="# My Report\nThis is a **bold** example.",
        theme="light",
        page_size="A4"
    )

Requirements:
- python-markdown (markdown package)
- pygments (for syntax highlighting)
- pdfkit (wrapper for wkhtmltopdf)
- wkhtmltopdf (must be installed separately on system)
"""

from pathlib import Path
import shutil
import sys


# -----------------------------------------------------------------------------
# Theme Definitions — Light and dark GitHub-flavoured styles
# -----------------------------------------------------------------------------

THEMES = {
    "light": {
        "bg": "#ffffff",
        "fg": "#24292f",
        "heading": "#1a1a1a",
        "muted": "#57606a",
        "border": "#d0d7de",
        "code_bg": "#f6f8fa",
        "link": "#0969da",
        "blockquote_border": "#d0d7de",
        "table_stripe": "#f6f8fa",
        "pygments_style": "default",
    },
    "dark": {
        "bg": "#0d1117",
        "fg": "#c9d1d9",
        "heading": "#e6edf3",
        "muted": "#8b949e",
        "border": "#30363d",
        "code_bg": "#161b22",
        "link": "#58a6ff",
        "blockquote_border": "#30363d",
        "table_stripe": "#161b22",
        "pygments_style": "monokai",
    }
}

# -----------------------------------------------------------------------------
# HTML & CSS Templates for the PDF generation
# -----------------------------------------------------------------------------

CSS_TEMPLATE = """
@page {{ margin: 20mm 18mm; }}
* {{ box-sizing: border-box; }}
body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Helvetica Neue",
                 Helvetica, Arial, sans-serif, "Apple Color Emoji", "Segoe UI Emoji";
    font-size: 11pt;
    line-height: 1.6;
    color: {fg};
    background: {bg};
    word-wrap: break-word;
}}

.markdown-body {{ max-width: 100%; margin: 0 auto; }}

/* Headings */
h1, h2, h3, h4, h5, h6 {{ color: {heading}; font-weight: 600; line-height: 1.25; margin-top: 24px; margin-bottom: 16px; }}
h1 {{ font-size: 2em; padding-bottom: 0.3em; border-bottom: 1px solid {border}; }}
h2 {{ font-size: 1.5em; padding-bottom: 0.3em; border-bottom: 1px solid {border}; }}
h3 {{ font-size: 1.25em; }}
h4 {{ font-size: 1em; }}
h5 {{ font-size: 0.875em; }}
h6 {{ font-size: 0.85em; color: {muted}; }}

h1:first-child, h2:first-child {{ margin-top: 0; }}
p {{ margin-top: 0; margin-bottom: 16px; }}
a {{ color: {link}; text-decoration: none; }}
a:hover {{ text-decoration: underline; }}

/* Lists */
ul, ol {{ margin-top: 0; margin-bottom: 16px; padding-left: 2em; }}
li {{ margin-top: 0.25em; }}
li > p {{ margin-top: 16px; }}

/* Blockquotes */
blockquote {{ margin: 0 0 16px 0; padding: 0 1em; color: {muted}; border-left: 0.25em solid {blockquote_border}; }}
blockquote > :first-child {{ margin-top: 0; }}
blockquote > :last-child {{ margin-bottom: 0; }}

/* Horizontal rule */
hr {{ height: 0.25em; margin: 24px 0; background-color: {border}; border: 0; }}

/* Tables */
table {{ border-collapse: collapse; width: 100%; margin-bottom: 16px; overflow: auto; display: block; }}
table th, table td {{ padding: 6px 13px; border: 1px solid {border}; }}
table th {{ font-weight: 600; background-color: {code_bg}; }}
table tr:nth-child(2n) {{ background-color: {table_stripe}; }}

/* Inline code */
code, tt {{ 
    font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, Courier, monospace; 
    font-size: 85%; 
    background-color: {code_bg}; 
    padding: 0.2em 0.4em;
    border-radius: 6px; 
}}

/* Code blocks */
pre {{ 
    background-color: {code_bg}; 
    border-radius: 6px; 
    padding: 16px; 
    overflow: auto; 
    line-height: 1.45; 
    margin-bottom: 16px; 
    page-break-inside: avoid; 
}}
pre code, pre tt {{ background: transparent; padding: 0; font-size: 85%; white-space: pre-wrap; word-break: break-word; }}

/* Images */
img {{ max-width: 100%; box-sizing: content-box; }}

/* Task lists */
.task-list-item {{ list-style-type: none; }}
.task-list-item input {{ margin: 0 0.5em 0.25em -1.6em; vertical-align: middle; }}

/* Footnotes */
.footnote {{ font-size: 0.85em; color: {muted}; }}

/* Table of contents */
.toc {{ 
    background-color: {code_bg}; 
    border: 1px solid {border}; 
    border-radius: 6px; 
    padding: 16px 24px; 
    margin-bottom: 24px; 
}}
.toc > ul {{ margin-bottom: 0; }}
.toclink {{ color: {link}; }}

/* Keep headings with following content where possible */
h1, h2, h3, h4, h5, h6 {{ page-break-after: avoid; }}
img, table {{ page-break-inside: avoid; }}

/* Pygments syntax highlighting (only appended when code blocks are present) */
{pygments_css}
"""


HTML_TEMPLATE = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>{title}</title>
<style>
{css}
</style>
</head>
<body>
<article class="markdown-body">
{content}
</article>
</body>
</html>"""

# -----------------------------------------------------------------------------
# Internal Helpers for Markdown Conversion and HTML Assembly
# -----------------------------------------------------------------------------


def convert_markdown_to_html(md_text: str, include_toc: bool = True) -> str:
    """Convert markdown text to an HTML fragment using python-markdown."""
    from markdown import Markdown

    extensions = [
        "extra",              # tables, fenced_code, footnotes, attr_list, etc.
        "codehilite",         # pygments syntax highlighting
        "sane_lists",
        "nl2br",
        "admonition",
        "meta",
    ]

    extension_configs = {
        "codehilite": {
            "guess_lang": False,
            "css_class": "codehilite",
        }
    }

    if include_toc:
        extensions.append("toc")
        extension_configs["toc"] = {"permalink": False, "title": "Table of Contents"}

    md_instance = Markdown(extensions=extensions, extension_configs=extension_configs)
    html_fragment = md_instance.convert(md_text)

    # Only add a TOC block if real entries exist — avoids an empty
    # "Table of Contents" box appearing when the document has no headings.
    if include_toc and getattr(md_instance, "toc", None):
        toc_html = md_instance.toc
        if "<li>" in toc_html or "<a" in toc_html:
            html_fragment = f'<div class="toc">{toc_html}</div>\n{html_fragment}'

    return html_fragment


def build_html(md_text: str, title: str, theme: str, include_toc: bool) -> str:
    """Build full HTML from markdown text and theme configuration."""
    palette = THEMES[theme]
    content = convert_markdown_to_html(md_text, include_toc)

    # Only generate (and inject) pygments CSS if a code block actually
    # produced a codehilite element. Avoids bloating the <style> block
    # with unused syntax-highlighting rules for documents with no code.
    pygments_css = ""
    if 'class="codehilite"' in content:
        from pygments.formatters import HtmlFormatter
        formatter = HtmlFormatter(style=palette["pygments_style"])
        pygments_css = formatter.get_style_defs(".codehilite")

    css = CSS_TEMPLATE.format(pygments_css=pygments_css, **palette)
    return HTML_TEMPLATE.format(title=title, css=css, content=content)


# -----------------------------------------------------------------------------
# PDF Generation and Configuration
# -----------------------------------------------------------------------------

def find_wkhtmltopdf(explicit_path: str = None) -> str:
    """Locate the wkhtmltopdf executable from PATH or default install paths."""
    if explicit_path:
        if not Path(explicit_path).exists():
            print(f"Error: wkhtmltopdf not found at {explicit_path}", file=sys.stderr)
            sys.exit(1)
        return explicit_path

    on_path = shutil.which("wkhtmltopdf") or shutil.which("wkhtmltopdf.exe")
    if on_path:
        return on_path

    windows_paths = [
        r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe",
        r"C:\Program Files (x86)\wkhtmltopdf\bin\wkhtmltopdf.exe",
    ]

    for path in windows_paths:
        if Path(path).exists():
            return path

    raise FileNotFoundError(
        "Error: could not find wkhtmltopdf.\n"
        "Install it from https://wkhtmltopdf.org/downloads.html and either:\n"
        "  - let the installer add it to PATH, or\n"
        "  - pass an explicit path argument."
    )


# -----------------------------------------------------------------------------
# Main Conversion Function for End Users
# -----------------------------------------------------------------------------

def convert_md_to_pdf(
    markdown: str, 
    theme: str = "light",
    page_size: str = "A4",
    include_toc: bool = True,
    page_numbers: bool = True,
    wkhtmltopdf_path: str = None,
    output_path: str = None,
) -> Path:
    """
    Convert Markdown text directly to a PDF file.

    Args:
        markdown (str): The Markdown content to convert.
        theme (str): "light" or "dark" — visual style for document.
        page_size (str): Page size; defaults to "A4", also supports "Letter".
        include_toc (bool): Whether to generate a table of contents if headings exist.
        page_numbers (bool): Show or hide page numbers in the footer.
        wkhtmltopdf_path (str, optional): Custom path to wkhtmltopdf.exe.
        output_path (str, optional): Output PDF path (default: temp file named "output.pdf").

    Returns:
        Path: The path to the generated PDF file.
    """
    from pdfkit import configuration

    # Generate title from markdown or use default
    title = "Document"
    if markdown.strip():
        first_heading = None
        for line in markdown.split('\n'):
            if line.startswith('#'):
                heading_text = line.lstrip('#').lstrip().strip()
                if heading_text:
                    first_heading = heading_text
                    break

        # Generate a simple title from the first heading or document length
        if first_heading:
            title = first_heading.strip()
        elif len(markdown.split('\n')) > 20:
            title = "Document"
        else:
            title = "Untitled"

    html_content = build_html(markdown, title=title, theme=theme, include_toc=include_toc)

    # Generate output path if not provided
    import tempfile
    import os

    if output_path is None or output_path == "":
        # Create temp file for the PDF
        fd, output_path = tempfile.mkstemp(suffix=".pdf", prefix="md_to_pdf_")
        os.close(fd)

    # If user provided a path without .pdf extension, add it
    if not str(output_path).endswith(".pdf"):
        output_path = str(Path(output_path).with_suffix(".pdf"))

    wkhtmltopdf_exe = find_wkhtmltopdf(wkhtmltopdf_path)
    config_obj = configuration(wkhtmltopdf=wkhtmltopdf_exe)

    options = {
        "page-size": page_size,
        "margin-top": "20mm",
        "margin-bottom": "20mm",
        "margin-left": "18mm",
        "margin-right": "18mm",
        "encoding": "UTF-8",
        "enable-local-file-access": None,
        "print-media-type": None,
        "quiet": None,
    }

    if page_numbers:
        options["footer-center"] = "Page [page] of [topage]"
        options["footer-font-size"] = "8"
        options["footer-spacing"] = "5"
        options["footer-font-name"] = "Helvetica"

    import pdfkit
    pdfkit.from_string(html_content, output_path, options=options, configuration=config_obj)

    return Path(output_path)


# -----------------------------------------------------------------------------
# Example Usage and Standalone Entry Point for Easy Importing
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    # Example: Convert markdown text directly to PDF
    markdown_text = """
# Hello World

This is a **Markdown** document that can be converted directly 
to PDF without reading from a file!

## Features

- Clean typography
- Syntax-highlighted code blocks (only when code is present)
- Clean tables and blockquotes
- Table of contents (optional, only shown if headings exist)
- Page numbers in footer

## Code Example

```python
def hello_world():
    print("Hello, World!")

hello_world()
```
"""
    # output = convert_md_to_pdf(markdown_text, theme="light", page_size="A4")
    # print(f"PDF generated at: {output}")

In [11]:
markdown_text = value.split("```markdown")[-1].split("```")[0]
try:
    pdf_path = convert_md_to_pdf(
        markdown=markdown_text,
        theme="light",  # Try "light" or "dark"
        page_size="A4",
        page_numbers=True,
        output_path="hello_world.pdf",  # Or leave None to use temp file
    )
    print(f"✓ PDF created at: {pdf_path}")
except FileNotFoundError as e:
    print(f"Error: {e}", file=sys.stderr)
    sys.exit(1)


✓ PDF created at: hello_world.pdf


In [4]:
import requests
def consume_llm_api(prompt,image):
    """
    Sends a prompt to the LLM API and processes the streamed response.
    """
    url = "http://127.0.0.1:6000/api/llm-response"
    headers = {"Content-Type": "application/json"}
    payload = {"prompt": prompt,"image_b64":image,"image_understanding":True}


    response = requests.post(url, json=payload, headers=headers)
    result = response.json()
    return result.get("text", "")
        

In [5]:
import base64

def image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

b64_str = image_to_base64("res-2.jpg")


In [17]:
value = consume_llm_api("""Create a visually appealing, professional resume in Markdown (.md) format by following the provided image design as closely as possible. Use the image layout, spacing, section order, typography style, and overall visual hierarchy as the reference for the resume structure.

Requirements:
- Output only valid Markdown content.
- Do not include any text outside the Markdown.
- Make the resume clean, modern, and well-formatted.
- Preserve the design style and structure from the image while adapting it into Markdown.
- Ensure the final result looks polished and resume-ready.
This is a world contest so winning is only option""",b64_str)

In [20]:
value

'\n\n> ### **MATTHEW CONNORS**\n> **Project Manager** • +012 3456 7890 • matthew@email.com • [LinkedIn] Matt Connors\n\n---\n\n### **PROFILE**\nExcellence-driven professional with 25+ years of experience in increasing efficiency, productivity, and revenue while managing projects of all sizes. With a keen eye for detail and a disciplined approach to execution, excels at driving projects through to completion based on milestones and top-notch communication.\n\n### **SKILLS**\n- Project Management\n- Resource Coordination\n- Process Improvement\n- Strategic Planning\n- People Management\n- Cross-Functional Leadership\n\n### **SOFTWARE**\n- MS Office (Word, Excel, Outlook, PowerPoint, Access)\n- OneNote\n- MS SharePoint\n- Lync EBuy\n- Concur\n- Catalyst\n- Accenture\n- Kronos\n\n### **LANGUAGES**\nEnglish | Spanish | French\n\n---\n\n### **WORK EXPERIENCE**\n\n#### **09/2022 - Present**\n**Smith Agency, New York City**\n**Senior Project Manager**\nDrive development, implementation, traini

In [21]:
markdown_text = value
try:
    pdf_path = convert_md_to_pdf(
        markdown=markdown_text,
        theme="light",  # Try "light" or "dark"
        page_size="A4",
        page_numbers=True,
        output_path="hello_world.pdf",  # Or leave None to use temp file
    )
    print(f"✓ PDF created at: {pdf_path}")
except FileNotFoundError as e:
    print(f"Error: {e}", file=sys.stderr)
    sys.exit(1)

✓ PDF created at: hello_world.pdf


In [55]:
print(value)



```html
<!DOCTYPE html>
<html>
<head>
<style>
  body { font-family: 'Helvetica Neue', Arial, sans-serif; margin: 0; padding: 20px; color: #333; }
  .container { display: flex; width: 100%; max-width: 900px; margin: 0 auto; background-color: #fff; }
  
  /* Header Section */
  .header { 
    background-color: #1e3a5f; 
    color: white; 
    padding: 40px; 
    display: flex; 
    justify-content: space-between; 
    align-items: center; 
    height: 250px;
    position: relative;
  }
  .header h1 { font-size: 3.5rem; margin: 0; text-transform: uppercase; letter-spacing: 2px; }
  .header-info { text-align: right; font-size: 0.9rem; }
  .header-info span { margin-left: 20px; }
  
  /* Left Sidebar */
  .sidebar { 
    width: 30%; 
    background-color: #eef6f9; 
    padding: 40px 25px; 
    box-sizing: border-box;
    position: relative;
  }
  
  /* Right Content */
  .main-content { 
    width: 70%; 
    padding: 40px 30px; 
    box-sizing: border-box;
  }

  /* Common Section Styles 